In [1]:
import pandas as pd
import numpy as np
import random
import tensorflow as tf
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler, QuantileTransformer, PowerTransformer
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping

# 1. Daten einlesen
df = pd.read_csv('training_data_ready.csv')
df = df.drop([21,43,92,159], axis=0)  # Falls es eine unnötige Index-Spalte gibt
target_cols = ['x', 'y', 'z', 'Gewicht'] 
x_cols = [col for col in df.columns if col not in target_cols + ['source_file','Objektname','Objektkategorie','OAufnahme','Schwierigkeit','Gewicht']]
    
X = df[x_cols]
y = df[target_cols]

# 2. Train-Test-Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scaler-Dictionary definieren
scaler_dict = {
    #'standard': StandardScaler(),
    #'minmax': MinMaxScaler(),
    'robust': RobustScaler(),
    'maxabs': MaxAbsScaler(),
    #'quantile': QuantileTransformer(output_distribution='normal', random_state=42),
    #'power': PowerTransformer(method='yeo-johnson')
}

# 4. Suchraum für die 100 Architekturen definieren
param_grid = {
    'scaler_name': ['robust', 'maxabs'], # ['standard', 'minmax', 'robust', 'maxabs', 'quantile', 'power']
    'hidden_layers': [
        [128, 64, 32],
        [256, 128, 64],
        [64, 32],
        [128, 64],
        [512, 256, 128, 64],
        [64, 64, 64],
        [32, 16]
    ],
    'l2_reg': [0.5, 0.2, 0.1, 0.01, 0.0],      # L2-Regularisierung (0.0 = deaktiviert)
    'dropout_rate': [0.0, 0.1, 0.2],    # Dropout-Rate (0.0 = deaktiviert)
    'learning_rate': [0.001, 0.0005, 0.01, 0.005, 0.05]   # Geschwindigkeit des Optimizers
}

# Generiere exakt 100 zufällige, einzigartige Kombinationen
num_combinations = 100
random_search_configs = list(ParameterSampler(param_grid, n_iter=num_combinations, random_state=42))

print(f"{len(random_search_configs)} Konfigurationen erfolgreich generiert.")

100 Konfigurationen erfolgreich generiert.


In [ ]:
# Variablen zum Tracking des besten Modells (für das finale Abspeichern)
best_val_loss = float('inf')
best_model = None
best_scaler_name = None

# Liste für die Gesamtauswertung aller Modelle
all_results = []

print(f"Starte die Suche über {len(random_search_configs)} Netze...")

for i, config in enumerate(random_search_configs):
    print(f"\n[{i+1}/{len(random_search_configs)}] Teste Konfiguration:")
    print(f" -> Scaler: {config['scaler_name']} | Netz: {config['hidden_layers']} | L2: {config['l2_reg']} | Dropout: {config['dropout_rate']} | LR: {config['learning_rate']}")
    
    # 1. Daten mit dem gewählten Scaler transformieren
    current_scaler = scaler_dict[config['scaler_name']]
    X_train_scaled = current_scaler.fit_transform(X_train)
    
    # 2. Modell dynamisch nach Konfiguration aufbauen
    model = models.Sequential()
    model.add(layers.Input(shape=(X_train_scaled.shape[1],)))
    
    for units in config['hidden_layers']:
        if config['l2_reg'] > 0:
            model.add(layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(config['l2_reg'])))
        else:
            model.add(layers.Dense(units, activation='relu'))
        
        if config['dropout_rate'] > 0:
            model.add(layers.Dropout(config['dropout_rate']))
            
    # Ausgabeschicht
    model.add(layers.Dense(4))
    
    # Optimizer konfigurieren
    optimizer = tf.keras.optimizers.Adam(learning_rate=config['learning_rate'])
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    
    # 3. Early Stopping
    early_stopping = EarlyStopping(
        monitor='val_loss', 
        patience=400, 
        restore_best_weights=True, 
        verbose=0
    )
    
    # 4. Training (500 Epochen)
    history = model.fit(
        X_train_scaled, y_train,
        epochs=2000,
        batch_size=8,
        validation_split=0.1,
        callbacks=[early_stopping],
        verbose=0
    )
    
    # 5. Besten Validierungsfehler auslesen
    min_val_loss = min(history.history['val_loss'])
    corresponding_mae = history.history['val_mae'][history.history['val_loss'].index(min_val_loss)]
    epochs_trained = len(history.history['val_loss'])
    
    print(f" -> Fertig nach {epochs_trained} Epochen. Bester Val-Loss: {min_val_loss:.4f}")
    
    # Ergebnisse in der Liste für das spätere Ranking speichern
    all_results.append({
        'scaler': config['scaler_name'],
        'layers': str(config['hidden_layers']),
        'l2': config['l2_reg'],
        'dropout': config['dropout_rate'],
        'lr': config['learning_rate'],
        'val_loss': min_val_loss,
        'val_mae': corresponding_mae,
        'epochs': epochs_trained,
        'model_object': model # Wir merken uns das Objekt, falls wir es später brauchen
    })
    
    # Das absolut beste Modell für den Schnellzugriff tracken
    if min_val_loss < best_val_loss:
        best_val_loss = min_val_loss
        best_model = model
        best_scaler_name = config['scaler_name']

print("\n================ ARCHITEKTURSUCHE BEENDET ================")

# 6. Ergebnisse nach val_loss aufsteigend sortieren (kleinerer Fehler = besser)
all_results_sorted = sorted(all_results, key=lambda x: x['val_loss'])

# 7. Die Top 5 Architekturen formatiert ausgeben
print("\nDIE TOP 5 BESTEN ARCHITEKTUREN:\n")
print(f"{'Platz':<6} | {'Scaler':<9} | {'Netz-Struktur':<18} | {'L2':<6} | {'Dropout':<7} | {'LR':<6} | {'Val-Loss (MSE)':<14} | {'Val-MAE':<8}")
print("-" * 95)

for rank, res in enumerate(all_results_sorted[:10], start=1):
    print(f"#{rank:<4} | {res['scaler']:<9} | {res['layers']:<18} | {res['l2']:<6} | {res['dropout']:<7} | {res['lr']:<6} | {res['val_loss']:<14.6f} | {res['val_mae']:<8.4f}")


Starte die Suche über 100 Netze...

[1/100] Teste Konfiguration:
 -> Scaler: robust | Netz: [128, 64, 32] | L2: 0.5 | Dropout: 0.1 | LR: 0.0005
 -> Fertig nach 2000 Epochen. Bester Val-Loss: 166.5320

[2/100] Teste Konfiguration:
 -> Scaler: maxabs | Netz: [32, 16] | L2: 0.01 | Dropout: 0.1 | LR: 0.05
 -> Fertig nach 1491 Epochen. Bester Val-Loss: 490.0318

[3/100] Teste Konfiguration:
 -> Scaler: maxabs | Netz: [64, 32] | L2: 0.01 | Dropout: 0.1 | LR: 0.01
 -> Fertig nach 755 Epochen. Bester Val-Loss: 393.6426

[4/100] Teste Konfiguration:
 -> Scaler: robust | Netz: [128, 64, 32] | L2: 0.01 | Dropout: 0.1 | LR: 0.05
 -> Fertig nach 578 Epochen. Bester Val-Loss: 163.8344

[5/100] Teste Konfiguration:
 -> Scaler: maxabs | Netz: [128, 64, 32] | L2: 0.01 | Dropout: 0.0 | LR: 0.001
 -> Fertig nach 932 Epochen. Bester Val-Loss: 391.0580

[6/100] Teste Konfiguration:
 -> Scaler: robust | Netz: [256, 128, 64] | L2: 0.0 | Dropout: 0.1 | LR: 0.0005
 -> Fertig nach 1218 Epochen. Bester Val-Loss:

In [ ]:
final_scaler = scaler_dict[best_scaler_name]
X_test_scaled = final_scaler.transform(X_test)
test_eval = best_model.evaluate(X_test_scaled, y_test)

# Optional: Speichere das beste Modell ab
best_model.save('best_neural_network_with_Weights.keras')

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 381.6526 - mae: 12.2205


In [ ]:
# save X_test_scaled to a file for later use
np.save('X_test_scaled.npy', X_test_scaled)
np.save('y_test.npy', y_test)
np.save('X_train_scaled.npy', X_train_scaled)
np.save('y_train.npy', y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)

y_pred_df = pd.DataFrame(
    y_pred,
    columns=['x_pred', 'y_pred', 'z_pred', 'Gewicht_pred'],
    index=y_test.index
)

results = y_test.copy()
results[['x_pred', 'y_pred', 'z_pred', 'Gewicht_pred']] = y_pred_df

# source_file Spalte hinzufügen
results['source_file'] = df.loc[y_test.index, 'source_file'].values

results['abs_error_x'] = np.abs(results['x_pred'] - results['x'])
results['abs_error_y'] = np.abs(results['y_pred'] - results['y'])
results['abs_error_z'] = np.abs(results['z_pred'] - results['z'])
results['abs_error_Gewicht'] = np.abs(results['Gewicht_pred'] - results['Gewicht'])
results['mse_x'] = (results['x_pred'] - results['x']) ** 2
results['mse_y'] = (results['y_pred'] - results['y']) ** 2
results['mse_z'] = (results['z_pred'] - results['z']) ** 2
results['mse_Gewicht'] = (results['Gewicht_pred'] - results['Gewicht']) ** 2

print("Auswertung Testdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MAE Gewicht: {results['abs_error_Gewicht'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")
print(f"  - MSE Gewicht: {results['mse_Gewicht'].mean():.4f}")

results_display = results[[
    'source_file',
    'x', 'x_pred', 'abs_error_x',
    'y', 'y_pred', 'abs_error_y',
    'z', 'z_pred', 'abs_error_z',
    'Gewicht', 'Gewicht_pred', 'abs_error_Gewicht'
]]
print("\nVorhersage vs. Realität für den Testdatensatz:")
print(results_display)

In [ ]:
y_pred = model.predict(X_train_scaled)

y_pred_df = pd.DataFrame(
    y_pred,
    columns=['x_pred', 'y_pred', 'z_pred'],
    index=y_train.index
)

results = y_train.copy()
results[['x_pred', 'y_pred', 'z_pred', 'Gewicht_pred']] = y_pred_df

# source_file Spalte hinzufügen
results['source_file'] = df.loc[y_train.index, 'source_file'].values

results['abs_error_x'] = np.abs(results['x_pred'] - results['x'])
results['abs_error_y'] = np.abs(results['y_pred'] - results['y'])
results['abs_error_z'] = np.abs(results['z_pred'] - results['z'])
results['abs_error_Gewicht'] = np.abs(results['Gewicht_pred'] - results['Gewicht'])

results['mse_x'] = (results['x_pred'] - results['x']) ** 2
results['mse_y'] = (results['y_pred'] - results['y']) ** 2
results['mse_z'] = (results['z_pred'] - results['z']) ** 2
results['mse_Gewicht'] = (results['Gewicht_pred'] - results['Gewicht']) ** 2

print("Auswertung Trainingsdatensatz:")
print(f"  - MAE x: {results['abs_error_x'].mean():.4f}")
print(f"  - MAE y: {results['abs_error_y'].mean():.4f}")
print(f"  - MAE z: {results['abs_error_z'].mean():.4f}")
print(f"  - MAE Gewicht: {results['abs_error_Gewicht'].mean():.4f}")
print(f"  - MSE x: {results['mse_x'].mean():.4f}")
print(f"  - MSE y: {results['mse_y'].mean():.4f}")
print(f"  - MSE z: {results['mse_z'].mean():.4f}")
print(f"  - MSE Gewicht: {results['mse_Gewicht'].mean():.4f}")

results_display = results[[
    'source_file',
    'x', 'x_pred', 'abs_error_x',
    'y', 'y_pred', 'abs_error_y',
    'z', 'z_pred', 'abs_error_z',
    'Gewicht', 'Gewicht_pred', 'abs_error_Gewicht'
]]
print("\nVorhersage vs. Realität für den Trainingsdatensatz:")
print(results_display)